# Paper 3 — Meta-Learned Adaptive Forecasting for Heterogeneous Food Price Time Series

**Research question.** Hundreds of food SKUs range from almost-never-changing to highly volatile. Should
they all share one forecasting model, or can a meta-learner, reading only a SKU's *time-series profile*,
allocate a model (or a weighted combination) that beats a single global choice?

**Design.**
* **Candidate pool (8 models):** Naive, seasonal naive (7), moving average (7), AutoETS, AutoARIMA,
  AutoTheta, global LightGBM, global N-BEATS-lite.
* **Meta-features** (27 per SKU, training window only): stickiness and intermittency (change rate, average
  inter-change interval ADI, CV^2 of change sizes — the Syntetos-Boylan classes), volatility, skewness,
  autocorrelations, trend and weekly-seasonality strength, spectral entropy, stability, lumpiness, flat-run
  length, recency and size of the last change, price level and range.
* **Meta-labels:** each model's loss on the **validation** window. Base models never use the validation
  window for fitting (internal early-stopping holdouts only), so meta-labels are genuinely out-of-sample.
* **Strategies:** A one global model; B per-SKU selection by own validation loss; C per-category
  selection; **D1** meta-classifier; **D2 (proposed)** meta-regression of every model's loss followed by
  a softmax-weighted combination (FFORMA-style); plus a simple average of all models and the test-set oracle
  (upper bound). D1 and D2 are evaluated with **product-level 5-fold cross-validation**, so the allocation for
  a SKU never uses that SKU's own validation losses.

**Outputs** under `food_price_research/paper3_meta_learning/`.

## 2. Dataset description

In [ ]:
# ==== 0. CONFIGURATION & GOOGLE DRIVE =====================================
PAPER = "paper3_meta_learning"
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
SMOKE = os.environ.get("FPR_SMOKE", "0") == "1"   # True -> tiny, fast end-to-end test run
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/food_price_research"
else:
    ROOT = os.environ.get("FPR_ROOT", os.path.abspath("food_price_research"))
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'statsforecast', 'lightgbm'], check=False)

from pathlib import Path
ROOT = Path(ROOT); RAW = ROOT / "raw"; RAW.mkdir(parents=True, exist_ok=True)
PREP_DIR = ROOT / "shared_prep"                      # cleaned panel, shared by all five papers
PAPER_DIR = ROOT / PAPER
FIG_DIR, TAB_DIR, RES_DIR, CKPT_DIR = [PAPER_DIR / s for s in ("figures", "tables", "results", "checkpoints")]
for d in (PREP_DIR, FIG_DIR, TAB_DIR, RES_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

cands = sorted(RAW.glob("[dD]ataset-2024.*"))
if not cands and IN_COLAB:
    from google.colab import files
    print("Upload dataset-2024.csv (or Dataset-2024.xlsx) - it is saved to", RAW)
    for k, v in files.upload().items():
        (RAW / k).write_bytes(v)
    cands = sorted(RAW.glob("[dD]ataset-2024.*"))
assert cands, f"Put dataset-2024.csv in {RAW}"
DATA_PATH = cands[0]
print("paper:", PAPER, "| data:", DATA_PATH, "| smoke:", SMOKE)

## 1. Shared data preparation (identical in all five notebooks)

The raw panel is a wide table (`Name | 2024-01-02 | ... | 2025-02-05`): 5,976 SKUs x 401 days, already
complete. Quality control, all estimated on training data only:

* **Scraper outages.** On 56 days not a single SKU changes price (the median day has ~80 changes): the scraper
  did not run and prices were carried forward. These cells are *imputed*, so they are excluded from every loss,
  metric target and graph statistic. The first day after an outage (a *gap-end* day) accumulates several days of
  changes and is excluded from co-movement statistics.
* **Spike-and-revert repair.** A jump of more than x1.5 that returns to within 5 % of the previous level within
  7 days is treated as a scrape error (wrong variant shown) and replaced by the previous level.
* **Variant breaks.** A persistent level shift of more than x3 within the training window is almost surely a
  re-used product slot; those SKUs are excluded.
* **Platform batch-update days.** Days whose number of simultaneous changes exceeds median + 5 MAD. Batch
  updates create spurious co-movement, so they are excluded from correlation-type statistics.
* **Category tagger.** Priority-ordered keyword rules on product names; non-food SKUs are kept as a control
  group and excluded from headline results.
* **Activity tiers** (on the earliest training window, so no look-ahead): T0 food SKUs that never change,
  T1 1-9 changes, T2 >= 10 changes. Headline results use the *active* universe T1 u T2.
* **Rolling-origin evaluation.** Three folds, each with a 30-day validation and a 30-day test window
  (test windows: 2024-11-08..12-07, 12-08..01-06, 2025-01-07..02-05). Models never see validation or test
  targets during fitting; forecasts at each origin use only data up to that origin.
* **Targets.** 100 x log(P[t+h] / P[t]) for h = 1..30; metrics are computed in price space at h in {1,3,7,14,30}.
  Because most prices are sticky we also report **CC-sMAPE** (sMAPE only where the price actually changes) and
  **change AUPRC**, alongside MAE, RMSE, MAPE, sMAPE, WAPE and MASE.

In [ ]:
# ============================================================================
# SECTION 1 — SHARED DATA PREPARATION (identical, deterministic in every notebook)
# ============================================================================
import re, json, time, math, pickle, hashlib, warnings, itertools
from pathlib import Path
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 1.1  Category tagger (rule-based, from product names; priority-ordered)
# ---------------------------------------------------------------------------
# Non-food rules are checked FIRST so that e.g. "hair oil" never becomes edible
# oil and "baby shampoo" never becomes baby food.
CATEGORY_RULES = [
    # ---- non-food -------------------------------------------------------
    ("NF: Personal Care", r"shampoo|conditioner|face ?wash|facewash|facial|soap|lotion|deodorant|deo |roll on|perfume|body spray|"
                          r"toothpaste|toothbrush|tooth ?brush|mouthwash|hair oil|hair color|hair colour|hair removal|hair treatment|"
                          r"serum|sunscreen|sun protect|talcum|cream(?!.*(biscuit|wafer|cake|cheese|ice cream|milk|roll|sandwich))|"
                          r"vaseline|petroleum jelly|nose strip|shaving|razor|blade|sanitary|napkin|pad[s]? |tampon|condom|lip ?balm|"
                          r"lipstick|nail|kajal|mascara|glycerine|cleansing|scrub|makeup|cosmetic|beauty|skin|aloe ?vera gel|"
                          r"hand ?wash|body wash|shower gel|cotton bud|wipes|essential oil|facial oil|massage oil|coconut oil hair|"
                          r"boroplus|moisturi|toner|face mask|mask|blackhead|dandruff|gatsby|gloss|hair gel|lip colou?r|hair|shave|sanitizer|baby wash|baby bath|\blip"),
    ("NF: Baby Care",     r"diaper|nappy|pant [smlx]|belt \d|feeding bottle|bottles cleanser|nipple|baby pillow|nima|teether|"
                          r"baby lotion|baby oil|baby soap|baby shampoo|baby powder|baby wipes|pacifier|romper|cot sheet|pants .*kg"),
    ("NF: Household",     r"clean(er|ing)|detergent|dish ?wash|washing|bleach|disinfect|toilet|harpic|lizol|air freshener|"
                          r"freshener|aerosol|mosquito|insect|coil|killer|repellent|tissue|napkin|foil|cling film|garbage|"
                          r"broom|mop|brush|sponge|scrubber|battery|bulb|light|candle|matchbox|match box|lighter|polish|"
                          r"glass cleaner|floor|mat |mat$|pillow|towel|bucket|container|organizer|box$|jar |bottle(?!.*(juice|drink|water|milk|sauce|oil))|"
                          r"plate|bowl|spoon|glass |mug|cup(?! noodles)|tray|chopping board|knife|rolling pin|leveler|grill|net$|pan |pot |kettle|"
                          r"flask|tiffin|lunch box|hanger|umbrella|finish household|powerball|fabric|softener|surf excel|rin |wheel |"
                          r"cat food|dog food|pet |\bcat\b|kitten|\bdog\b|smartheart|pro diet|whiskas|pedigree|kitty|wrap|mixer|apron|strainer|duster|dish cover|holder|litter|"
                          r"spray|cockroach|\bhit\b|diesel|\bmobil\b|lubricant|\bcover\b|sheet"),
    ("NF: Health",        r"tablet|capsule|syrup(?!.*(chocolate|strawberry))|vitamin|antiseptic|dettol liquid|savlon|bandage|"
                          r"plaster|thermometer|oral saline|ors|pain relief|balm|inhaler|medicine|n95|surgical|glucose(?! biscuit)|"
                          r"test kit|pregnancy"),
    ("NF: Stationery & Misc", r"pencil|pen |pen$|ball ?pen|marker|highlighter|eraser|sharpener|notebook|note ?book|file |folder|"
                          r"stapler|punch|scissor|cutter|glue|tape|calculator|compass|geometry|ruler|a4|paper|envelope|sticker|"
                          r"flag|toy|puzzle|ball |badminton|shuttlecock|cricket|football|game|doll|remote|charger|cable|"
                          r"earphone|headphone|usb|mobile|phone|fusion set|moistener|clip|board|diary|book|colour pencil|crayon|"
                          r"gift|decoration|balloon|travel|bag|wallet|\bcard\b|\bbear\b|\bcar\b|instrument|acrylic|glitter|sash|tasbih|"
                          r"\bbat\b|lace|drawing|flower"),
    # ---- food ---------------------------------------------------------------
    ("Baby Food",         r"formula|infant|baby food|cerelac|nan optipro|lactogen|biomil|growing up milk|\(0-6|\(6-12|\(1-3|"
                          r"nido 1|nido 3|cereal.*(tin|baby)|mother's smile"),
    ("Beverages",         r"juice|drink|beverage|electrolyte|soda|cola|7 ?up|sprite|fanta|pepsi|coca|mojo|speed|energy|water|lassi|milk shake|"
                          r"shake|tang|horlicks|boost|ovaltine|bournvita|malt|squash|syrup|can$|smoothie|mocktail|aloe drink|frooto|frutika"),
    ("Tea & Coffee",      r"\btea\b|tea bag|green tea|coffee|nescafe|infusion|chai"),
    ("Dairy & Eggs",      r"milk|cheese|butter(?!.*(cookie|biscuit|peanut))|yogurt|yoghurt|\bdoi\b|curd|cream cheese|paneer|"
                          r"ghee|egg|dim\b|dairy|uht|evaporated|condensed|ice cream|kulfi"),
    ("Edible Oil",        r"soybean oil|soyabean|mustard oil|sunflower oil|palm oil|rice bran|olive oil|canola|"
                          r"vegetable oil|cooking oil|sesame oil|\boil\b"),
    ("Fish & Seafood",    r"fish|ilish|hilsha|rui|rohu|katla|pangas|tilapia|shing|magur|koi|pabda|chingri|shrimp|prawn|"
                          r"crab|squid|baila|boal|shol|taki|poa|rupchanda|pomfret|loitta|kachki|mola|tengra|shutki|dried fish"),
    ("Meat & Poultry",    r"beef|mutton|goat|lamb|chicken|broiler|deshi murgi|murgi|duck(?! eggs)|pigeon|quail|meat|"
                          r"liver|gizzard|keema|mince|bone in|boneless|drumstick|breast|wings|sausage|salami|nugget|patty|kebab"),
    ("Frozen & Ready",     r"frozen|ready to cook|ready mix|paratha|puri|samosa|singara|spring roll|haleem mix|biryani mix|"
                          r"instant|french fries|burger|pizza(?! sauce)"),
    ("Rice & Grains",     r"\brice\b|chinigura|basmati|miniket|nazirshail|atop|kalijira|polao|poha|chira|muri|khoi|oats|"
                          r"cornflakes|corn flakes|cereal|muesli|barley|quinoa|semai|shemai|laccha|lachcha|vermicelli|wheat"),
    ("Flour & Bakery",    r"atta|maida|flour|suji|semolina|besan|bread|bun|cake|rusk|toast|bakorkhani|bakery|dough|yeast|"
                          r"baking|pastry|croissant|donut|ruti|chapati|roti"),
    ("Lentils & Pulses",  r"\bdal\b|lentil|masur|moong|mung|mashkalai|chola|chick ?pea|kabuli|anchor dal|khesari|"
                          r"motor|peas|beans?\b|rajma"),
    ("Spices & Condiments", r"masala|spice|powder|chili|chilli|turmeric|holud|coriander|cumin|zeera|jeera|cardamom|elach|"
                          r"cinnamon|daruchini|clove|lobongo|black pepper|golmorich|bay leaf|tejpata|nutmeg|jayfal|mace|"
                          r"fenugreek|methi|panch phoron|mix$|salt|vinegar|flakes|seasoning|stock|soup|bouillon|gelatine|"
                          r"food colou?r|essence|baking soda|kewra|rose water|seed|herbs|china grass|mix\s*$"),
    ("Sugar & Sweeteners", r"sugar|chini|\bgur\b|jaggery|honey|mishri|sweetener|candy|chocolate|toffee|lollipop|jelly|"
                          r"marshmallow|gum|mint"),
    ("Snacks & Biscuits", r"biscuit|cookie|cracker|wafer|chips|chanachur|namkeen|dalmoth|bhujia|nuts|peanut(?! butter)|"
                          r"cashew|kaju|almond|pista|raisin|kismis|dates|khejur|popcorn|puffs|roll$|snack|bar |bar$|"
                          r"chanachur|jhal muri|sticks|crisps|pringles|kurkure|lays|badam"),
    ("Sauces & Spreads",  r"sauce|ketchup|mayonnaise|mayo|\bjam\b|jelly|spread|pickle|achar|chutney|peanut butter|nutella|"
                          r"dressing|paste|puree|olive(?! oil)"),
    ("Noodles & Pasta",   r"noodle|ramen|pasta|macaroni|spaghetti|fusilli|penne|maggi|cup noodles"),
    ("Fruits",            r"mango|aam|banana|kola\b|apple|orange|malta|grape|guava|peyara|papaya|pape|pineapple|anaros|"
                          r"watermelon|tormuj|lemon|lebu|lime|pomegranate|anar|litchi|lichu|jackfruit|kathal|coconut|"
                          r"narikel|dates|berry|kiwi|pear|plum|prune|wood apple|kodbel|amra|jambura|sofeda|fruit"),
    ("Vegetables",        r"potato|alu|aloo|onion|peyaj|piyaj|garlic|roshun|ginger|ada\b|tomato|brinjal|begun|cucumber|"
                          r"shosha|pumpkin|kumra|gourd|lau|korola|potol|dherosh|okra|cabbage|cauliflower|kopi|carrot|gajor|"
                          r"radish|mula|spinach|shak|sak\b|lettuce|capsicum|green chili|kacha morich|beans|borboti|"
                          r"mushroom|coriander leaves|dhonia pata|mint leaves|pudina|vegetable|sobji|corn|bhutta|"
                          r"betel|shupari|paan|chal kumra|kochu|arum|shim|motorshuti|lemongrass|drumstick|broccoli|\bpui\b|microgreen|sheem"),
]
NONFOOD_PREFIX = "NF: "
_COMPILED = [(c, re.compile(p, re.I)) for c, p in CATEGORY_RULES]

def fix_mojibake(s: str) -> str:
    """Repair UTF-8 read as cp1252 (e.g. 'Â±50 gm' -> '±50 gm')."""
    try:
        t = s.encode("cp1252").decode("utf-8")
        return t
    except Exception:
        return s.replace("Â±", "±").replace("Â ", " ").replace("â€™", "'")

def tag_category(name: str) -> str:
    n = " " + name.lower().strip() + " "
    for cat, rx in _COMPILED:
        if rx.search(n):
            return cat
    return "Unclassified"

# ---------------------------------------------------------------------------
# 1.2  Load the wide panel  (Name | date1 | date2 | ...)
# ---------------------------------------------------------------------------
def load_panel(path):
    path = Path(path)
    if path.suffix.lower() in (".xlsx", ".xls"):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path, encoding="utf-8-sig")
    df = df.rename(columns={df.columns[0]: "Name"})
    date_cols = [c for c in df.columns[1:]]
    dates = pd.to_datetime([str(c)[:10] for c in date_cols])
    P = df[date_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)
    names = df["Name"].astype(str).map(fix_mojibake).str.strip().tolist()
    return names, dates, P

# ---------------------------------------------------------------------------
# 1.3  Data-quality: interpolated-window mask, spike-and-revert repair,
#      persistent variant breaks, platform batch-update days
# ---------------------------------------------------------------------------
def detect_carried_forward_days(P, max_changes=2):
    """Scraper-outage days. When the scraper did not run, the panel carries the
    previous day's prices forward, so NOT A SINGLE product changes price that
    day (the median day has ~80 changes). These cells are imputed, not observed:
    they are excluded from every loss/metric target and from graph statistics.
    The first observed day after an outage accumulates several days of changes
    (a 'gap-end' day) and is flagged separately."""
    ch = np.concatenate([[10**9], (np.diff(P, axis=1) != 0).sum(axis=0)])
    cf = ch <= max_changes
    gap_end = np.zeros_like(cf)
    gap_end[1:] = cf[:-1] & ~cf[1:]
    return cf, gap_end

def repair_spikes(P, tau=np.log(1.5), max_len=7, tol=0.05):
    """Spike-and-revert repair: a jump |dlog| > tau that returns to within `tol`
    of the pre-jump level within `max_len` days is treated as a scrape error
    (wrong variant / pack size shown for a few days) and replaced by the
    pre-jump level. Returns cleaned matrix + per-series repair counts."""
    L = np.log(P)
    Q = P.copy(); n_rep = np.zeros(P.shape[0], dtype=int)
    T = P.shape[1]
    for i in range(P.shape[0]):
        x = L[i]
        d = np.diff(x)
        cand = np.where(np.abs(d) > tau)[0]
        t_skip = -1
        for j in cand:
            if j <= t_skip: continue
            base = x[j]
            for k in range(j + 1, min(j + 1 + max_len, T - 1)):
                if abs(x[k + 1] - base) <= tol:
                    Q[i, j + 1: k + 1] = P[i, j]
                    n_rep[i] += 1; t_skip = k
                    break
    return Q, n_rep

def detect_variant_breaks(P, t_end, thr=np.log(3.0)):
    """Persistent level shifts > x3 inside the training window: almost surely a
    re-used product slot (different pack/variant), not a price change."""
    d = np.abs(np.diff(np.log(P[:, : t_end + 1]), axis=1))
    return (d > thr).any(axis=1)

def detect_batch_days(P, t_end, k_mad=5.0):
    """Days where the number of simultaneous price changes is an outlier
    (median + k*MAD, threshold estimated on training days only)."""
    ch = (np.diff(P, axis=1) != 0).sum(axis=0)          # changes on day t (t>=1)
    ch = np.concatenate([[0], ch])
    tr = ch[1: t_end + 1]
    med = np.median(tr); mad = np.median(np.abs(tr - med)) + 1e-9
    thr = med + k_mad * 1.4826 * mad
    return ch > thr, ch, thr

# ---------------------------------------------------------------------------
# 1.4  Rolling-origin splits (3 folds, 30-day val + 30-day test each)
# ---------------------------------------------------------------------------
def make_folds(T, n_folds=3, test_len=30, val_len=30):
    folds = []
    for k in range(n_folds):
        test_end = T - 1 - test_len * (n_folds - 1 - k)
        test_start = test_end - test_len + 1
        val_end = test_start - 1; val_start = val_end - val_len + 1
        folds.append(dict(fold=k, train_end=val_start - 1, val_start=val_start, val_end=val_end,
                          test_start=test_start, test_end=test_end))
    return folds

# ---------------------------------------------------------------------------
# 1.5  Master prep (cached to Drive; identical in all five notebooks)
# ---------------------------------------------------------------------------
PREP_VERSION = "v1.1"

def build_prep(data_path, cache_dir, clean_spikes=True, force=False):
    cache_dir = Path(cache_dir); cache_dir.mkdir(parents=True, exist_ok=True)
    f = cache_dir / f"prep_{PREP_VERSION}.pkl"
    if f.exists() and not force:
        with open(f, "rb") as fh:
            prep = pickle.load(fh)
        print(f"[prep] loaded cached panel from {f}")
        return prep
    t0 = time.time()
    names, dates, P_raw = load_panel(data_path)
    N, T = P_raw.shape
    assert np.isfinite(P_raw).all() and (P_raw > 0).all(), "panel must be complete & positive"
    folds = make_folds(T)
    tier_end = folds[0]["train_end"]                    # leak-free: earliest training window
    cf_day, gap_end = detect_carried_forward_days(P_raw)
    P_clean, n_rep = repair_spikes(P_raw) if clean_spikes else (P_raw.copy(), np.zeros(N, int))
    variant_break = detect_variant_breaks(P_clean, tier_end)
    batch_day, daily_changes, batch_thr = detect_batch_days(P_clean, tier_end)
    cats = [tag_category(n) for n in names]
    is_food = np.array([not c.startswith(NONFOOD_PREFIX) and c != "Unclassified" for c in cats])
    # activity statistics on the tier window only (no look-ahead)
    dlog_tr = np.diff(np.log(P_clean[:, : tier_end + 1]), axis=1)
    n_changes = (dlog_tr != 0).sum(axis=1)
    vol = dlog_tr.std(axis=1)
    usable = is_food & ~variant_break
    tier = np.full(N, "T0", dtype=object)
    tier[usable & (n_changes >= 1)] = "T1"
    tier[usable & (n_changes >= 10)] = "T2"
    # volatility terciles among T1 (training-window std of log-returns)
    vol_bucket = np.full(N, "none", dtype=object)
    m = usable & (n_changes >= 1)
    q1, q2 = np.quantile(vol[m], [1 / 3, 2 / 3])
    vol_bucket[m & (vol <= q1)] = "low"
    vol_bucket[m & (vol > q1) & (vol <= q2)] = "medium"
    vol_bucket[m & (vol > q2)] = "high"
    meta = pd.DataFrame(dict(name=names, category=cats, is_food=is_food, variant_break=variant_break,
                             n_changes_train=n_changes, vol_train=vol, tier=tier, vol_bucket=vol_bucket,
                             spike_repairs=n_rep, mean_price_train=P_clean[:, : tier_end + 1].mean(axis=1)))
    prep = dict(names=names, dates=dates, P_raw=P_raw, P=P_clean, logP=np.log(P_clean), meta=meta,
                folds=folds, tier_end=tier_end, cf_day=cf_day, gap_end=gap_end,
                batch_day=batch_day, daily_changes=daily_changes, batch_thr=batch_thr,
                version=PREP_VERSION, built=time.strftime("%Y-%m-%d %H:%M"))
    with open(f, "wb") as fh:
        pickle.dump(prep, fh)
    print(f"[prep] built in {time.time()-t0:.1f}s and cached to {f}")
    return prep

def subset_idx(prep, tiers=("T1", "T2"), food_only=True, smoke_n=None, seed=0):
    meta = prep["meta"]
    m = meta.tier.isin(tiers).to_numpy()
    if food_only: m &= meta.is_food.to_numpy()
    idx = np.where(m)[0]
    if smoke_n is not None and len(idx) > smoke_n:
        rng = np.random.default_rng(seed); idx = np.sort(rng.choice(idx, smoke_n, replace=False))
    return idx

In [ ]:
# ============================================================================
# SECTION 1b — SHARED UTILITIES: figures, tables, checkpoints, windows, metrics
# ============================================================================
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats as sps

# ---------------------------------------------------------------------------
# Figure style (journal): 8-pt sans, thin marks, recessive grid, fixed palette
# ---------------------------------------------------------------------------
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
GREY = "#8a8985"
COL1, COL2 = 3.5, 7.2                       # single / double column widths (inches)
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8, "legend.fontsize": 7,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.6, "axes.edgecolor": "#52514e", "axes.grid": True, "grid.color": "#e4e3df",
    "grid.linewidth": 0.5, "lines.linewidth": 1.4, "lines.markersize": 4, "legend.frameon": False,
    "axes.prop_cycle": matplotlib.cycler(color=PALETTE), "pdf.fonttype": 42, "ps.fonttype": 42,
})

def savefig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"{name}.{ext}")
    print(f"[fig] {name} -> {FIG_DIR}")
    plt.show(); plt.close(fig)

# ---------------------------------------------------------------------------
# Tables: CSV + LaTeX (booktabs), no jinja2 dependency; optional bolding of best
# ---------------------------------------------------------------------------
def _fmt(v, nd):
    if isinstance(v, (float, np.floating)):
        if not np.isfinite(v): return "--"
        return f"{v:.{nd}f}"
    return str(v).replace("_", r"\_").replace("&", r"\&").replace("%", r"\%")

def save_table(df, name, caption="", label=None, nd=3, bold_min=None, bold_max=None, index=False, bold_row_min=None):
    """Write <name>.csv and <name>.tex (booktabs). bold_min/bold_max: list of
    columns in which the best value is set in bold."""
    df = df.reset_index() if index else df.copy()
    df.to_csv(TAB_DIR / f"{name}.csv", index=False)
    best = {}
    for c in (bold_min or []):
        if c in df: best[c] = pd.to_numeric(df[c], errors="coerce").min()
    for c in (bold_max or []):
        if c in df: best[c] = pd.to_numeric(df[c], errors="coerce").max()
    cols = list(df.columns)
    align = "".join("l" if not np.issubdtype(df[c].dtype, np.number) else "r" for c in cols)
    lines = [r"\begin{table}[t]", r"\centering", r"\small"]
    if caption: lines.append(rf"\caption{{{caption}}}")
    lines.append(rf"\label{{{label or 'tab:' + name}}}")
    lines += [rf"\begin{{tabular}}{{{align}}}", r"\toprule",
              " & ".join(_fmt(c, nd) for c in cols) + r" \\", r"\midrule"]
    for _, r in df.iterrows():
        cells = []
        rmin = pd.to_numeric(r[bold_row_min], errors="coerce").min() if bold_row_min else None
        for c in cols:
            s = _fmt(r[c], nd)
            if np.issubdtype(df[c].dtype, np.integer): s = str(int(r[c]))
            if bold_row_min and c in bold_row_min and np.isfinite(r[c]) and np.isclose(r[c], rmin):
                s = rf"\textbf{{{s}}}"
            elif c in best and isinstance(r[c], (float, np.floating, int, np.integer)) and np.isfinite(r[c]) \
               and np.isclose(r[c], best[c]):
                s = rf"\textbf{{{s}}}"
            cells.append(s)
        lines.append(" & ".join(cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    (TAB_DIR / f"{name}.tex").write_text("\n".join(lines))
    print(f"[table] {name} -> {TAB_DIR}")
    try:
        display(df)
    except NameError:
        print(df.to_string())
    return df

# ---------------------------------------------------------------------------
# Result caching & checkpoints (everything resumable after a Colab disconnect)
# ---------------------------------------------------------------------------
def cached(name, fn, force=False):
    f = RES_DIR / f"{name}.pkl"
    if f.exists() and not force:
        with open(f, "rb") as fh: return pickle.load(fh)
    out = fn()
    with open(f, "wb") as fh: pickle.dump(out, fh)
    return out

def set_seed(s):
    import random
    random.seed(s); np.random.seed(s)
    try:
        import torch
        torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ImportError:
        pass

# ---------------------------------------------------------------------------
# Feature tensor & rolling windows
#   channels: 0 = 100*log(P_t'/P_origin)   (filled at gather time)
#             1 = price-changed indicator  2 = observed (not carried-forward)
#             3,4 = day-of-week sin/cos     5 = log1p(days since last change)/5
#   targets : 100*log(P_{o+h}/P_o), h = 1..H, clipped to [-100, 100]
# ---------------------------------------------------------------------------
H_MAX, H_EVAL = 30, [1, 3, 7, 14, 30]
N_CH = 6

def build_features(prep, idx):
    lp = prep["logP"][idx]
    n, T = lp.shape
    chg = np.zeros_like(lp); chg[:, 1:] = (np.diff(lp, axis=1) != 0)
    obs = np.broadcast_to((~prep["cf_day"]).astype(np.float64), (n, T))
    dow = prep["dates"].dayofweek.to_numpy()
    s, c = np.sin(2 * np.pi * dow / 7), np.cos(2 * np.pi * dow / 7)
    dsc = np.zeros_like(lp)
    for t in range(1, T):
        dsc[:, t] = np.where(chg[:, t] > 0, 0, dsc[:, t - 1] + 1)
    F = np.stack([lp, chg, obs, np.broadcast_to(s, (n, T)), np.broadcast_to(c, (n, T)),
                  np.log1p(dsc) / 5], axis=-1).astype(np.float32)
    return F                                                   # (n, T, C); channel 0 = raw log-price

def origins_for(fold, part, L, H=H_MAX):
    """Forecast origins o (inclusive) for 'train' | 'val' | 'test'. Targets at
    o+1..o+H are masked beyond the segment end (no peeking)."""
    if part == "train":
        return np.arange(L - 1, fold["train_end"]), fold["train_end"]
    if part == "val":
        return np.arange(fold["val_start"] - 1, fold["val_end"]), fold["val_end"]
    return np.arange(fold["test_start"] - 1, fold["test_end"]), fold["test_end"]

def gather_windows(F, idx, origins, L, H, seg_end, cf_day, rows=None):
    """Vectorised, ORIGIN-MAJOR gather for products `rows` (local indices into F,
    default all) at forecast origins `origins`.
    Returns X (k, n, L, C), Y (k, n, H), M (k, n, H) and meta (k, n, 2) holding
    (global product id, origin day). Channel 0 of X is re-based to the origin."""
    rows = np.arange(F.shape[0]) if rows is None else np.asarray(rows)
    oo, rr = np.meshgrid(origins, rows, indexing="ij")                    # (k, n)
    tt = oo[..., None] + np.arange(-L + 1, 1)                              # (k, n, L)
    X = F[rr[..., None], np.maximum(tt, 0)].copy()
    base = F[rr, oo, 0]
    X[..., 0] = 100.0 * (X[..., 0] - base[..., None])
    hh = oo[..., None] + np.arange(1, H + 1)                               # (k, n, H)
    valid = hh <= seg_end
    hc = np.minimum(hh, F.shape[1] - 1)
    M = valid & ~cf_day[hc]
    Y = 100.0 * (F[rr[..., None], hc, 0] - base[..., None])
    Y = np.clip(np.where(M, Y, 0.0), -100, 100).astype(np.float32)
    meta = np.stack([np.asarray(idx)[rr], oo], -1)
    return X.astype(np.float32), Y, M, meta

# ---------------------------------------------------------------------------
# Evaluation in PRICE space. Predictions are 100*log-ratio vs the origin price.
# ---------------------------------------------------------------------------
def price_scale(prep, idx, fold):
    """MASE scale: mean |P_t - P_{t-1}| on the fold's training window."""
    P = prep["P"][idx, : fold["train_end"] + 1]
    return np.maximum(np.abs(np.diff(P, axis=1)).mean(axis=1), 1e-6)

def evaluate(pred, Y, M, meta, prep, fold, horizons=H_EVAL):
    """pred, Y, M: (..., H) arrays (any leading shape); meta (..., 2).
    Returns (summary per horizon, {h: per-product mean sAPE})."""
    H = Y.shape[-1]
    pred, Y, M, meta = pred.reshape(-1, H), Y.reshape(-1, H), M.reshape(-1, H), meta.reshape(-1, 2)
    P0 = prep["P"][meta[:, 0], meta[:, 1]]
    gids = np.unique(meta[:, 0])
    scale = pd.Series(price_scale(prep, gids, fold), index=gids)
    sc = scale.loc[meta[:, 0]].to_numpy()
    rows = []
    per_prod = {}
    for h in horizons:
        m = M[:, h - 1]
        if m.sum() == 0: continue
        y = P0[m] * np.exp(Y[m, h - 1] / 100); yh = P0[m] * np.exp(pred[m, h - 1] / 100)
        e = yh - y; ae = np.abs(e)
        sape = 200 * ae / (np.abs(y) + np.abs(yh))
        chg = np.abs(Y[m, h - 1]) > 1e-9
        prod = meta[m, 0]
        df = pd.DataFrame(dict(prod=prod, ae=ae, sape=sape, se=ae**2, ape=100 * ae / y, sc_ae=ae / sc[m], y=y))
        g = df.groupby("prod")
        mase_p = g.sc_ae.mean()
        rows.append(dict(h=h, n=int(m.sum()), MAE=ae.mean(), RMSE=np.sqrt((ae**2).mean()),
                         MAPE=df.ape.mean(), sMAPE=sape.mean(), WAPE=100 * ae.sum() / y.sum(),
                         MASE=mase_p.mean(), MASE_med=mase_p.median(),
                         CC_sMAPE=sape[chg].mean() if chg.any() else np.nan,
                         change_rate=chg.mean()))
        per_prod[h] = g.sape.mean()
    return pd.DataFrame(rows), per_prod

def change_auprc(score, Y, M, h):
    """AUPRC for 'will the price differ from today's price at horizon h?'."""
    from sklearn.metrics import average_precision_score
    H = Y.shape[-1]; score, Y, M = score.reshape(-1, H), Y.reshape(-1, H), M.reshape(-1, H)
    m = M[:, h - 1]; lab = np.abs(Y[m, h - 1]) > 1e-9
    if lab.sum() == 0 or lab.all(): return np.nan
    return average_precision_score(lab, score[m, h - 1])

# ---------------------------------------------------------------------------
# Statistical tests
# ---------------------------------------------------------------------------
def dm_test(l1, l2, h=1):
    """Diebold-Mariano with Harvey-Leybourne-Newbold small-sample correction.
    l1, l2: loss series (e.g. cross-sectional mean sAPE per origin). Negative
    statistic => model 1 more accurate."""
    d = np.asarray(l1) - np.asarray(l2); d = d[np.isfinite(d)]
    T = len(d)
    if T < 5: return np.nan, np.nan
    dbar = d.mean(); g = [np.mean((d[k:] - dbar) * (d[: T - k] - dbar)) for k in range(min(h, T - 1))]
    v = (g[0] + 2 * sum(g[1:])) / T
    if v <= 0: return np.nan, np.nan
    stat = dbar / np.sqrt(v) * np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T)
    return stat, 2 * sps.t.sf(abs(stat), df=T - 1)

def wilcoxon_paired(a, b):
    a, b = pd.Series(a), pd.Series(b); j = a.index.intersection(b.index)
    d = (a[j] - b[j]).to_numpy()
    if np.allclose(d, 0): return np.nan
    return sps.wilcoxon(d, zero_method="zsplit").pvalue

def bootstrap_ci(d, B=2000, seed=0, alpha=0.05):
    d = np.asarray(d); d = d[np.isfinite(d)]
    rng = np.random.default_rng(seed)
    bs = rng.choice(d, (B, len(d)), replace=True).mean(axis=1)
    return d.mean(), np.quantile(bs, alpha / 2), np.quantile(bs, 1 - alpha / 2)

def holm(pvals):
    p = np.asarray(pvals, dtype=float); out = np.full_like(p, np.nan)
    ok = np.isfinite(p); idx = np.argsort(p[ok]); m = ok.sum()
    adj = np.maximum.accumulate((m - np.arange(m)) * p[ok][idx]); adj = np.minimum(adj, 1)
    tmp = np.empty(m); tmp[idx] = adj; out[ok] = tmp
    return out

def per_origin_loss(pred, Y, M, meta, prep, h):
    """Cross-sectional mean sAPE per origin at horizon h (DM-test input)."""
    H = Y.shape[-1]
    pred, Y, M, meta = pred.reshape(-1, H), Y.reshape(-1, H), M.reshape(-1, H), meta.reshape(-1, 2)
    m = M[:, h - 1]
    P0 = prep["P"][meta[m, 0], meta[m, 1]]
    y = P0 * np.exp(Y[m, h - 1] / 100); yh = P0 * np.exp(pred[m, h - 1] / 100)
    s = pd.Series(200 * np.abs(yh - y) / (y + yh)).groupby(meta[m, 1]).mean()
    return s

# ---------------------------------------------------------------------------
# Dataset description (figure + table used in the Data section of every paper)
# ---------------------------------------------------------------------------
def dataset_overview(prep):
    meta, dates = prep["meta"], prep["dates"]
    rows = [("Products (SKUs)", len(meta)), ("Days", len(dates)),
            ("Period", f"{dates[0].date()} to {dates[-1].date()}"),
            ("Food SKUs (tagged)", int(meta.is_food.sum())),
            ("Excluded: variant breaks", int(meta.variant_break.sum())),
            ("Scraper-outage days (carried forward)", int(prep["cf_day"].sum())),
            ("Platform batch-update days", int(prep["batch_day"].sum())),
            ("Spike-and-revert repairs", int(meta.spike_repairs.sum())),
            ("T0 (food, never changes in train)", int(((meta.tier == "T0") & meta.is_food).sum())),
            ("T1 (food, 1-9 changes)", int((meta.tier == "T1").sum())),
            ("T2 (food, >=10 changes)", int((meta.tier == "T2").sum()))]
    save_table(pd.DataFrame(rows, columns=["Item", "Value"]).astype({"Value": str}), "data_summary",
               "Dataset summary after quality control.")
    fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.2))
    ch = prep["daily_changes"]
    ax[0].bar(dates, ch, width=1.0, color=PALETTE[0], lw=0)
    ax[0].bar(dates[prep["batch_day"]], ch[prep["batch_day"]], width=2.5, color=PALETTE[1], lw=0, label="batch update")
    ax[0].scatter(dates[prep["cf_day"]], np.zeros(prep["cf_day"].sum()), marker="|", s=30, color=GREY, label="outage (0 changes)")
    ax[0].set_title("(a) Daily number of price changes"); ax[0].legend(loc="upper left")
    ax[0].tick_params(axis="x", rotation=30)
    nc = meta.loc[meta.is_food, "n_changes_train"]
    ax[1].hist(np.log10(nc + 1), bins=30, color=PALETTE[0], edgecolor="white", lw=0.4)
    ax[1].set_xlabel("log$_{10}$(1 + price changes in training window)"); ax[1].set_ylabel("Food SKUs")
    ax[1].set_title("(b) Price stickiness")
    act = meta[meta.tier.isin(["T1", "T2"])]
    order = act.category.value_counts()
    ax[2].barh(order.index[::-1], order.values[::-1], color=PALETTE[0])
    ax[2].set_title("(c) Active food SKUs by category"); ax[2].tick_params(axis="y", labelsize=6)
    fig.tight_layout(); savefig(fig, "fig_data_overview")

# ---------------------------------------------------------------------------
# Fold data bundle: windows for train / val / test in one call (cached in RAM)
# ---------------------------------------------------------------------------
def fold_data(prep, F, idx, fold, L, H=H_MAX, parts=("train", "val", "test"), train_stride=1):
    out = {}
    for part in parts:
        O, seg_end = origins_for(fold, part, L, H)
        if part == "train": O = O[::train_stride]
        out[part] = gather_windows(F, idx, O, L, H, seg_end, prep["cf_day"])
    return out

def naive_family(X, H=H_MAX):
    """Naive (0), seasonal-naive(7) and moving-average(7) forecasts from windows.
    X channel 0 is 100*log(P/P_origin) so the naive forecast is identically 0."""
    k = X[..., 0]
    naive = np.zeros(k.shape[:-1] + (H,), np.float32)
    L = k.shape[-1]
    snaive = np.stack([k[..., L - 7 + ((h - 1) % 7)] for h in range(1, H + 1)], -1)
    ma = np.repeat(k[..., -7:].mean(-1, keepdims=True), H, -1)
    return {"Naive": naive, "SNaive-7": snaive.astype(np.float32), "MA-7": ma.astype(np.float32)}

def results_frame(res_dict):
    """{model: summary-df} -> long frame with a 'model' column."""
    return pd.concat([d.assign(model=m) for m, d in res_dict.items()], ignore_index=True)

In [ ]:
# ============================================================================
# SECTION 1c — SHARED PYTORCH UTILITIES: resumable trainer + building blocks
# ============================================================================
import torch, torch.nn as nn, torch.nn.functional as Fnn
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
print("device:", DEVICE)

def masked_huber(pred, y, m, delta=1.0):
    m = m.float()
    l = Fnn.huber_loss(pred, y, reduction="none", delta=delta)
    return (l * m).sum() / m.sum().clamp(min=1.0)

def iterate_flat(X, Y, M, bs, shuffle, rng=None, extra=None):
    """Mini-batches over flattened (origin, product) windows. X: (k,n,L,C).
    extra: dict of (k,n,...) arrays gathered alongside (e.g. category ids)."""
    k, n = X.shape[:2]
    N = k * n
    order = (rng or np.random.default_rng(0)).permutation(N) if shuffle else np.arange(N)
    Xf, Yf, Mf = X.reshape(N, *X.shape[2:]), Y.reshape(N, -1), M.reshape(N, -1)
    ex = {kk: v.reshape(N, *v.shape[2:]) for kk, v in (extra or {}).items()}
    for s in range(0, N, bs):
        j = order[s: s + bs]
        b = dict(x=torch.from_numpy(Xf[j]).to(DEVICE), y=torch.from_numpy(Yf[j]).to(DEVICE),
                 m=torch.from_numpy(Mf[j]).to(DEVICE))
        for kk, v in ex.items(): b[kk] = torch.from_numpy(v[j]).to(DEVICE)
        yield b

def iterate_snap(X, Y, M, bs, shuffle, rng=None, extra=None):
    """Mini-batches of whole cross-sections (all products at an origin) for
    graph / cross-product models. X: (k, n, L, C) -> batch x: (b, n, L, C)."""
    k = X.shape[0]
    order = (rng or np.random.default_rng(0)).permutation(k) if shuffle else np.arange(k)
    for s in range(0, k, bs):
        j = order[s: s + bs]
        b = dict(x=torch.from_numpy(X[j]).to(DEVICE), y=torch.from_numpy(Y[j]).to(DEVICE),
                 m=torch.from_numpy(M[j]).to(DEVICE))
        for kk, v in (extra or {}).items(): b[kk] = torch.from_numpy(v[j]).to(DEVICE)
        yield b

def fit(model, train_iter, val_iter, loss_fn, ckpt_path, epochs=30, lr=1e-3, wd=1e-4,
        patience=6, verbose=True, clip=1.0):
    """Resumable training loop. `train_iter(epoch)` / `val_iter()` yield batches,
    `loss_fn(model, batch)` returns a scalar loss. A checkpoint is written after
    every epoch (model, optimiser, scheduler, history, early-stopping state);
    re-running the cell resumes where it stopped; a finished run is skipped."""
    ckpt_path = Path(ckpt_path); ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    st = dict(epoch=0, best=np.inf, bad=0, hist=[], done=False, best_state=None)
    if ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        st = ck["state"]
        if st["done"]:
            model.load_state_dict(st["best_state"]); model.eval()
            return model, pd.DataFrame(st["hist"])
        if verbose: print(f"  resuming {ckpt_path.name} at epoch {st['epoch']}")
    for ep in range(st["epoch"], epochs):
        model.train(); t0 = time.time(); tl, nb = 0.0, 0
        for b in train_iter(ep):
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=AMP, dtype=torch.float16):
                loss = loss_fn(model, b)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(opt); scaler.update(); tl += loss.item(); nb += 1
        sched.step()
        model.eval(); vl, vb = 0.0, 0
        with torch.no_grad():
            for b in val_iter():
                with torch.autocast(DEVICE.type, enabled=AMP, dtype=torch.float16):
                    vl += loss_fn(model, b).item(); vb += 1
        vl /= max(vb, 1); tl /= max(nb, 1)
        st["hist"].append(dict(epoch=ep, train=tl, val=vl, sec=time.time() - t0))
        if vl < st["best"] - 1e-5:
            st["best"], st["bad"] = vl, 0
            st["best_state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            st["bad"] += 1
        st["epoch"] = ep + 1
        stop = st["bad"] >= patience or st["epoch"] >= epochs
        st["done"] = stop
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), state=st), ckpt_path)
        if verbose: print(f"  ep {ep:02d} train {tl:.4f} val {vl:.4f} ({time.time()-t0:.1f}s)")
        if stop: break
    if st["best_state"] is not None: model.load_state_dict(st["best_state"])
    model.eval()
    return model, pd.DataFrame(st["hist"])

@torch.no_grad()
def predict(model, batches, key="x", fwd=None):
    model.eval(); out = []
    for b in batches:
        with torch.autocast(DEVICE.type, enabled=AMP, dtype=torch.float16):
            p = fwd(model, b) if fwd else model(b[key])
        out.append(p.float().cpu().numpy())
    return np.concatenate(out, 0)

# ---------------------------------------------------------------------------
# Building blocks shared by several papers
# ---------------------------------------------------------------------------
class PatchEncoder(nn.Module):
    """PatchTST-style temporal encoder: (B, L, C) -> (B, d)."""
    def __init__(self, L, C, d=64, patch=6, layers=2, heads=4, dropout=0.1):
        super().__init__()
        assert L % patch == 0
        self.patch, self.np = patch, L // patch
        self.proj = nn.Linear(patch * C, d)
        self.pos = nn.Parameter(torch.randn(1, self.np, d) * 0.02)
        el = nn.TransformerEncoderLayer(d, heads, 2 * d, dropout, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(el, layers, enable_nested_tensor=False)
        self.out = nn.Sequential(nn.LayerNorm(2 * d), nn.Linear(2 * d, d))
    def tokens(self, x):
        B, L, C = x.shape
        z = x.reshape(B, self.np, self.patch * C)
        return self.enc(self.proj(z) + self.pos)                 # (B, np, d)
    def forward(self, x):
        h = self.tokens(x)
        return self.out(torch.cat([h.mean(1), h[:, -1]], -1))

class GRUEncoder(nn.Module):
    def __init__(self, C, d=64, layers=1):
        super().__init__(); self.rnn = nn.GRU(C, d, layers, batch_first=True)
    def forward(self, x): return self.rnn(x)[0][:, -1]

class LSTMEncoder(nn.Module):
    def __init__(self, C, d=64, layers=2, dropout=0.1):
        super().__init__(); self.rnn = nn.LSTM(C, d, layers, batch_first=True, dropout=dropout)
    def forward(self, x): return self.rnn(x)[0][:, -1]

class MLPEncoder(nn.Module):
    def __init__(self, L, C, d=64):
        super().__init__(); self.net = nn.Sequential(nn.Flatten(), nn.Linear(L * C, 2 * d), nn.GELU(), nn.Linear(2 * d, d))
    def forward(self, x): return self.net(x)

class Head(nn.Module):
    def __init__(self, d, H, dropout=0.1):
        super().__init__(); self.net = nn.Sequential(nn.LayerNorm(d), nn.Dropout(dropout), nn.Linear(d, d), nn.GELU(), nn.Linear(d, H))
    def forward(self, h): return self.net(h)

In [ ]:
# ==== 1d. BUILD / LOAD THE CLEAN PANEL (cached on Drive) ====================
prep = build_prep(DATA_PATH, PREP_DIR)
meta, FOLDS = prep["meta"], prep["folds"]
CATS = sorted(meta.category.unique()); CAT2ID = {c: i for i, c in enumerate(CATS)}
for f in FOLDS:
    d = prep["dates"]
    print(f"fold {f['fold']}: train ..{d[f['train_end']].date()} | val {d[f['val_start']].date()}..{d[f['val_end']].date()}"
          f" | test {d[f['test_start']].date()}..{d[f['test_end']].date()}")
print(meta.tier.value_counts().to_dict())

In [ ]:
# ==== 2. DATASET OVERVIEW ===================================================
dataset_overview(prep)

In [ ]:
# ==== 3. CONFIGURATION ======================================================
import lightgbm as lgb
from joblib import Parallel, delayed
from sklearn.model_selection import KFold
L, H = 60, H_MAX
IDX = subset_idx(prep, ("T1", "T2"), smoke_n=60 if SMOKE else None)
N = len(IDX)
CAT = np.array([CAT2ID[c] for c in meta.category.to_numpy()[IDX]])
F = build_features(prep, IDX)
FOLD_IDS = [1, 2] if SMOKE else [0, 1, 2]
EPOCHS = 2 if SMOKE else 40
POOL = ["Naive", "SNaive-7", "MA-7", "AutoETS", "AutoARIMA", "AutoTheta", "LightGBM", "N-BEATS-lite"]
D = {f: fold_data(prep, F, IDX, FOLDS[f], L) for f in FOLD_IDS}
print(f"N={N} | folds {FOLD_IDS} | pool {POOL}")

def cat_extra(X): return np.ascontiguousarray(np.broadcast_to(CAT[None], X.shape[:2]))

## 4. Base forecasts on the validation and test windows

In [ ]:
# ==== 4.1 STATISTICAL MODELS (fit on train, filtered forward through val + test) ====
def _stat_one(y, train_end, origins, H, kind):
    from statsforecast.models import AutoARIMA, AutoETS, AutoTheta
    try:
        mdl = {"AutoARIMA": lambda: AutoARIMA(season_length=7), "AutoETS": lambda: AutoETS(season_length=7),
               "AutoTheta": lambda: AutoTheta(season_length=7)}[kind]().fit(y[: train_end + 1])
        out = np.stack([mdl.forward(y=y[: o + 1], h=H)["mean"] - y[o] for o in origins])
    except Exception:
        out = np.zeros((len(origins), H))
    return 100 * np.clip(out, -1, 1)

BASE = {f: {} for f in FOLD_IDS}                     # BASE[f][model] = (val_pred, test_pred), each (k, n, H)
for f in FOLD_IDS:
    fold = FOLDS[f]
    Ov, _ = origins_for(fold, "val", L); Ot, _ = origins_for(fold, "test", L)
    O = np.concatenate([Ov, Ot]); lp = prep["logP"][IDX]
    for kind in ("AutoETS", "AutoARIMA", "AutoTheta"):
        def run():
            r = Parallel(n_jobs=-1)(delayed(_stat_one)(lp[i], fold["train_end"], O, H, kind) for i in range(N))
            return np.stack(r, 1).astype(np.float32)
        t0 = time.time(); P = cached(f"stat_{kind}_fold{f}_N{N}", run)
        BASE[f][kind] = (P[: len(Ov)], P[len(Ov):]); print(f"fold {f} {kind}: {time.time()-t0:.0f}s")
    nv, nt = naive_family(D[f]["val"][0]), naive_family(D[f]["test"][0])
    for m in nv: BASE[f][m] = (nv[m], nt[m])

In [ ]:
# ==== 4.2 GLOBAL LightGBM + N-BEATS-lite (early stopping on an internal training holdout) ====
HOLD = 30                                            # last 30 training origins = internal early-stopping set

def tab_features(X, cat):
    v = X[..., 0]; c = X[..., 1]
    feats = [v[..., -30:-1][..., ::-1]]
    for w in (7, 14, 30, 60):
        feats += [v[..., -w:].mean(-1, keepdims=True), v[..., -w:].std(-1, keepdims=True)]
    feats += [c[..., -7:].sum(-1, keepdims=True), c[..., -30:].sum(-1, keepdims=True), c.sum(-1, keepdims=True),
              X[..., -1, 5:6], X[..., -1, 3:5], X[..., -1, 2:3], cat[..., None].astype(np.float32)]
    Z = np.concatenate(feats, -1); return Z.reshape(-1, Z.shape[-1])

def lgbm_base(f, max_rows=400_000, seed=0):
    def run():
        X, Y, M, _ = D[f]["train"]
        Xa, Ya, Ma = X[:-HOLD], Y[:-HOLD], M[:-HOLD]; Xh, Yh, Mh = X[-HOLD:], Y[-HOLD:], M[-HOLD:]
        Za, Zh = tab_features(Xa, cat_extra(Xa)), tab_features(Xh, cat_extra(Xh))
        outs = []
        for part in ("val", "test"):
            Xp = D[f][part][0]; outs.append((tab_features(Xp, cat_extra(Xp)), np.zeros(Xp.shape[:2] + (H,), np.float32)))
        rng = np.random.default_rng(seed)
        for h in H_EVAL:
            j = np.where(Ma[..., h - 1].ravel())[0]
            if len(j) > max_rows: j = rng.choice(j, max_rows, replace=False)
            mh = Mh[..., h - 1].ravel()
            reg = lgb.LGBMRegressor(n_estimators=100 if SMOKE else 2000, learning_rate=0.03, num_leaves=63, min_child_samples=50,
                                    subsample=0.8, subsample_freq=1, colsample_bytree=0.8, objective="huber", alpha=1.0,
                                    random_state=seed, verbose=-1)
            reg.fit(Za[j], Ya[..., h - 1].ravel()[j], eval_set=[(Zh[mh], Yh[..., h - 1].ravel()[mh])],
                    callbacks=[lgb.early_stopping(100, verbose=False)])
            for Z, out in outs: out[..., h - 1] = reg.predict(Z).reshape(out.shape[:2])
        # fill intermediate horizons by linear interpolation between reported horizons
        for _, out in outs:
            for a, b in zip(H_EVAL[:-1], H_EVAL[1:]):
                for h in range(a + 1, b):
                    w = (h - a) / (b - a); out[..., h - 1] = (1 - w) * out[..., a - 1] + w * out[..., b - 1]
        return outs[0][1], outs[1][1]
    return cached(f"lgbm_base_fold{f}_N{N}", run)

class NBeatsLite(nn.Module):
    """Generic N-BEATS: 3 fully-connected blocks with backcast/forecast residual stacking on the
    price and change channels, plus a category embedding."""
    def __init__(self, n_cat, H=H_MAX, width=256, blocks=3):
        super().__init__()
        self.blocks = nn.ModuleList([nn.Sequential(nn.Linear(2 * L + 16, width), nn.ReLU(), nn.Linear(width, width), nn.ReLU(),
                                                   nn.Linear(width, 2 * L + H)) for _ in range(blocks)])
        self.cat = nn.Embedding(n_cat, 16)
    def forward(self, x, cat):
        r = torch.cat([x[..., 0], x[..., 1]], -1); c = self.cat(cat); fc = 0
        for b in self.blocks:
            o = b(torch.cat([r, c], -1)); r = r - o[:, : 2 * L]; fc = fc + o[:, 2 * L:]
        return fc

def nbeats_base(f, seed=0):
    def run():
        X, Y, M, _ = D[f]["train"]
        tr = (X[:-HOLD], Y[:-HOLD], M[:-HOLD]); ho = (X[-HOLD:], Y[-HOLD:], M[-HOLD:])
        set_seed(seed); model = NBeatsLite(len(CATS)); rng = np.random.default_rng(seed)
        lf = lambda m, b: masked_huber(m(b["x"], b["cat"]), b["y"], b["m"])
        model, _ = fit(model, lambda ep: iterate_flat(*tr, 512, True, rng, {"cat": cat_extra(tr[0])}),
                       lambda: iterate_flat(*ho, 2048, False, extra={"cat": cat_extra(ho[0])}),
                       lf, CKPT_DIR / f"nbeats__f{f}__s{seed}.pt", EPOCHS, 1e-3, patience=6, verbose=False)
        outs = []
        for part in ("val", "test"):
            Xp, Yp, Mp, _ = D[f][part]
            outs.append(predict(model, iterate_flat(Xp, Yp, Mp, 2048, False, extra={"cat": cat_extra(Xp)}),
                                fwd=lambda m, b: m(b["x"], b["cat"])).reshape(Yp.shape))
        return tuple(outs)
    return cached(f"nbeats_base_fold{f}_N{N}", run)

for f in FOLD_IDS:
    t0 = time.time(); BASE[f]["LightGBM"] = lgbm_base(f); BASE[f]["N-BEATS-lite"] = nbeats_base(f)
    print(f"fold {f} global models: {time.time()-t0:.0f}s")

## 5. Meta-features (training window only)

In [ ]:
# ==== 5. META-FEATURES ======================================================
def spectral_entropy(x):
    p = np.abs(np.fft.rfft(x - x.mean())) ** 2; p = p[1:]
    if p.sum() <= 0: return 0.0
    p = p / p.sum(); return float(-(p * np.log(p + 1e-12)).sum() / np.log(len(p)))

def acf(x, k):
    x = x - x.mean(); d = (x * x).sum()
    return float((x[k:] * x[:-k]).sum() / d) if d > 0 else 0.0

def meta_features(f):
    te = FOLDS[f]["train_end"]; dates = prep["dates"][: te + 1]
    rows = []
    for i in IDX:
        y = prep["logP"][i, : te + 1]; obs = ~prep["cf_day"][1: te + 1]
        r = np.diff(y)[obs]; nz = r[r != 0]; ch_days = np.where(np.diff(y) != 0)[0]
        n = len(nz); t = np.arange(len(y))
        lin = np.polyfit(t, y, 1); quad = np.polyfit(t, y, 2)
        fit_lin = np.polyval(lin, t); r2 = 1 - ((y - fit_lin) ** 2).sum() / max(((y - y.mean()) ** 2).sum(), 1e-12)
        dow = dates.dayofweek.to_numpy()[1:][obs]
        seas = pd.Series(r).groupby(dow).mean().var() / max(r.var(), 1e-12) if r.var() > 0 else 0.0
        roll = pd.Series(y).rolling(30)
        flat = np.diff(np.r_[-1, ch_days, len(y) - 1]).max() / len(y)
        rows.append(dict(
            n_changes=n, change_rate=n / max(len(r), 1), ADI=len(r) / max(n, 1),
            CV2=(np.abs(nz).std() / np.abs(nz).mean()) ** 2 if n > 1 else 0.0,
            mean_abs_change=np.abs(nz).mean() if n else 0.0, vol=r.std(), max_abs_ret=np.abs(r).max(),
            skew=float(sps.skew(nz)) if n > 2 else 0.0, up_share=(nz > 0).mean() if n else 0.5,
            acf1=acf(r, 1), acf2=acf(r, 2), acf7=acf(r, 7), trend_r2=r2, slope=lin[0] * 100, curvature=quad[0] * 1e4,
            weekly_seasonality=seas, spectral_entropy=spectral_entropy(r),
            stability=float(roll.mean().var()), lumpiness=float(roll.var().var()),
            longest_flat_frac=flat, days_since_change=(len(y) - 1 - ch_days[-1]) if len(ch_days) else len(y),
            last_change=float(np.diff(y)[ch_days[-1]]) if len(ch_days) else 0.0,
            n_changes_30d=int((np.diff(y)[-30:] != 0).sum()), log_price=float(y.mean()), log_range=float(y.max() - y.min()),
            changes_on_batch_days=float(prep["batch_day"][1: te + 1][np.diff(y) != 0].mean()) if len(ch_days) else 0.0,
            category=CAT[list(IDX).index(i)]))
    return pd.DataFrame(rows, index=IDX)

PHI = {f: cached(f"phi_fold{f}_N{N}", lambda: meta_features(f)) for f in FOLD_IDS}
FEATS = [c for c in PHI[FOLD_IDS[0]].columns]
save_table(PHI[FOLD_IDS[-1]].describe().T[["mean", "std", "min", "50%", "max"]].reset_index().rename(columns={"index": "feature"}),
           "tab_meta_features", "Meta-features (latest fold, training window).", nd=3)

## 6. Meta-labels and strategies

In [ ]:
# ==== 6. LOSS MATRICES + STRATEGIES =========================================
def loss_matrix(pred, part, f):
    """Per-product mean sAPE over origins and reported horizons -> (n,)."""
    X, Y, M, mt = D[f][part]
    _, per = evaluate(pred, Y, M, mt, prep, FOLDS[f])
    return pd.concat(per, axis=1).mean(1).reindex(IDX)

LV = {f: pd.DataFrame({m: loss_matrix(BASE[f][m][0], "val", f) for m in POOL}) for f in FOLD_IDS}
LT = {f: pd.DataFrame({m: loss_matrix(BASE[f][m][1], "test", f) for m in POOL}) for f in FOLD_IDS}

def combine(f, W):
    """W: (n, M) weights (rows sum to 1) -> test forecast (k, n, H)."""
    P = np.stack([BASE[f][m][1] for m in POOL], -1)
    return (P * W[None, :, None, :]).sum(-1).astype(np.float32)

def onehot(choice):
    W = np.zeros((N, len(POOL))); W[np.arange(N), [POOL.index(c) for c in choice]] = 1; return W

TAU_GRID = [0.02, 0.05, 0.1, 0.2, 0.5, 1.0]
def meta_models(f, n_splits=5, seed=0):
    """Product-level CV. D1: LightGBM classifier of the best model. D2: LightGBM regressors of each
    model's (log) validation loss -> softmax(-pred / tau) weights, tau chosen by inner CV."""
    X = PHI[f][FEATS].to_numpy(dtype=float); lv = LV[f].to_numpy(); logl = np.log(lv + 1e-3)
    best = lv.argmin(1)
    W1, W2 = np.zeros_like(lv), np.zeros_like(lv); shap = np.zeros((len(POOL), X.shape[1]))
    params = dict(n_estimators=50 if SMOKE else 400, learning_rate=0.03, num_leaves=15, min_child_samples=10,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1, random_state=seed)
    for tr, te in KFold(n_splits, shuffle=True, random_state=seed).split(X):
        classes = np.unique(best[tr])
        if len(classes) > 1:
            clf = lgb.LGBMClassifier(**params).fit(X[tr], best[tr])
            W1[np.ix_(te, classes)] = clf.predict_proba(X[te])
        else:
            W1[te, classes[0]] = 1
        preds_tr, preds_te = np.zeros((len(tr), len(POOL))), np.zeros((len(te), len(POOL)))
        for j in range(len(POOL)):
            reg = lgb.LGBMRegressor(**params).fit(X[tr], logl[tr, j])
            preds_te[:, j] = reg.predict(X[te])
            preds_tr[:, j] = reg.predict(X[tr])
            shap[j] += np.abs(reg.predict(X[te], pred_contrib=True)[:, :-1]).mean(0) / n_splits
        # tau: minimise the combined *validation* loss proxy on the training products
        def comb_loss(pred_logl, lvals, tau):
            z = -(pred_logl - pred_logl.min(1, keepdims=True)) / tau; w = np.exp(z); w /= w.sum(1, keepdims=True)
            return (w * lvals).sum(1).mean()
        tau = min(TAU_GRID, key=lambda t: comb_loss(preds_tr, lv[tr], t))
        z = -(preds_te - preds_te.min(1, keepdims=True)) / tau; w = np.exp(z); W2[te] = w / w.sum(1, keepdims=True)
    D1 = np.zeros_like(W1); D1[np.arange(N), W1.argmax(1)] = 1
    return D1, W2, pd.DataFrame(shap, index=POOL, columns=FEATS)

STRAT, SHAP, CHOICE = {f: {} for f in FOLD_IDS}, {}, {}
for f in FOLD_IDS:
    lv, lt = LV[f], LT[f]
    gbest = lv.mean().idxmin()
    STRAT[f]["A: global best (" + gbest + ")"] = onehot([gbest] * N)
    STRAT[f]["B: per-SKU validation selection"] = onehot(lv.idxmin(1).tolist())
    cat_best = lv.groupby(CAT).mean().idxmin(1)
    STRAT[f]["C: per-category selection"] = onehot([cat_best[c] for c in CAT])
    D1, W2, SHAP[f] = meta_models(f)
    STRAT[f]["D1: meta-classifier"] = D1
    STRAT[f]["D2: meta-weighted combination (proposed)"] = W2
    STRAT[f]["Simple average of pool"] = np.full((N, len(POOL)), 1 / len(POOL))
    STRAT[f]["Oracle (test-set selection)"] = onehot(lt.idxmin(1).tolist())
    CHOICE[f] = dict(val_best=lv.idxmin(1), test_best=lt.idxmin(1), d1=pd.Series(np.array(POOL)[D1.argmax(1)], index=IDX))
    for m in POOL: STRAT[f][f"single: {m}"] = onehot([m] * N)

## 7. Results

In [ ]:
# ==== 7. MAIN TABLE =========================================================
def strat_name(s): return s.split(" (")[0] if s.startswith("A:") else s
def eval_strategy(sname):
    fs, pp = [], []
    for f in FOLD_IDS:
        key = [k for k in STRAT[f] if strat_name(k) == strat_name(sname)][0]
        pred = combine(f, STRAT[f][key]); te = D[f]["test"]
        s, per = evaluate(pred, te[1], te[2], te[3], prep, FOLDS[f])
        s["AUPRC_chg"] = [change_auprc(np.abs(pred), te[1], te[2], h) for h in s.h]
        fs.append(s); pp.append(pd.concat(per, axis=1).mean(1))
    return pd.concat(fs).groupby("h").mean(numeric_only=True), pd.concat(pp).groupby(level=0).mean()

ALL = [strat_name(k) for k in STRAT[FOLD_IDS[0]]]
EVS = {s: eval_strategy(s) for s in ALL}
rows = []
orc = EVS["Oracle (test-set selection)"][1]
for s in ALL:
    mu, pp = EVS[s]
    r = dict(strategy=s)
    for h in H_EVAL: r[f"sMAPE@{h}"] = mu.loc[h, "sMAPE"]
    r.update(sMAPE=mu.sMAPE.mean(), CC_sMAPE=mu.CC_sMAPE.mean(), WAPE=mu.WAPE.mean(), MASE=mu.MASE.mean(),
             mean_regret=(pp - orc).mean(), AUPRC_chg7=mu.loc[7, "AUPRC_chg"])
    rows.append(r)
MAIN = pd.DataFrame(rows)
MAIN["group"] = np.where(MAIN.strategy.str.startswith("single"), "single model", "strategy")
MAIN = MAIN.sort_values(["group", "sMAPE"], ascending=[False, True]).drop(columns="group")
save_table(MAIN, "tab_strategies", "Model-allocation strategies vs single models (test, mean over rolling folds). "
           "mean_regret = mean per-SKU sMAPE excess over the test-set oracle.", nd=3,
           bold_min=[f"sMAPE@{h}" for h in H_EVAL] + ["sMAPE", "CC_sMAPE", "WAPE", "MASE", "mean_regret"])

In [ ]:
# ==== 7.1 SIGNIFICANCE (proposed vs every other strategy) ===================
P = EVS["D2: meta-weighted combination (proposed)"][1]
rows = []
for s in ALL:
    if s.startswith("D2") or s.startswith("Oracle"): continue
    b = EVS[s][1]; j = P.index.intersection(b.index)
    mean, lo, hi = bootstrap_ci((b[j] - P[j]).values)
    rows.append(dict(vs=s, d_sMAPE=mean, CI_lo=lo, CI_hi=hi, p_Wilcoxon=wilcoxon_paired(P, b)))
sig = pd.DataFrame(rows); sig["p_holm"] = holm(sig.p_Wilcoxon)
save_table(sig, "tab_significance", "D2 vs alternatives: d_sMAPE = alternative minus D2 per-SKU sMAPE "
           "(positive favours D2), bootstrap 95% CI, Wilcoxon p (Holm).", nd=4)

In [ ]:
# ==== 7.2 SELECTION ACCURACY + CONFUSION MATRIX =============================
rows = []
for f in FOLD_IDS:
    c = CHOICE[f]
    rows.append(dict(fold=f, B_val_selection_acc=(c["val_best"] == c["test_best"]).mean(),
                     D1_meta_classifier_acc=(c["d1"] == c["test_best"]).mean(),
                     majority_baseline_acc=(c["test_best"] == c["test_best"].mode()[0]).mean()))
save_table(pd.DataFrame(rows), "tab_selection_accuracy", "Share of SKUs for which a strategy picks the test-optimal model.", nd=3)
f = FOLD_IDS[-1]
cm = pd.crosstab(CHOICE[f]["test_best"], CHOICE[f]["d1"]).reindex(index=POOL, columns=POOL, fill_value=0)
cmn = cm.div(cm.sum(1).replace(0, np.nan), axis=0)
fig, ax = plt.subplots(figsize=(COL1, 3.0))
im = ax.imshow(cmn.values, cmap="Blues", vmin=0, vmax=1); ax.grid(False)
ax.set_xticks(range(len(POOL)), POOL, rotation=45, ha="right"); ax.set_yticks(range(len(POOL)), POOL)
for i in range(len(POOL)):
    for j in range(len(POOL)):
        if cm.values[i, j]: ax.text(j, i, cm.values[i, j], ha="center", va="center", fontsize=6,
                                    color="white" if cmn.values[i, j] > 0.6 else "#0b0b0b")
ax.set_xlabel("D1 choice"); ax.set_ylabel("Test-optimal model"); ax.set_title("Meta-classifier confusion (latest fold)")
fig.colorbar(im, ax=ax, shrink=0.8); fig.tight_layout(); savefig(fig, "fig_confusion")

In [ ]:
# ==== 7.3 FIGURES: best-model share by category; regret CDF; feature space ====
f = FOLD_IDS[-1]
share = pd.crosstab(meta.loc[IDX, "category"].to_numpy(), LV[f].idxmin(1).to_numpy(), normalize="index").reindex(columns=POOL, fill_value=0)
fig, ax = plt.subplots(figsize=(COL2 * 0.62, 3.4))
im = ax.imshow(share.values, cmap="Blues", vmin=0, vmax=1, aspect="auto"); ax.grid(False)
ax.set_xticks(range(len(POOL)), POOL, rotation=45, ha="right"); ax.set_yticks(range(len(share)), share.index, fontsize=6)
fig.colorbar(im, ax=ax, label="share of SKUs", shrink=0.8)
ax.set_title("Validation-best model by category"); fig.tight_layout(); savefig(fig, "fig_best_model_by_category")

fig, ax = plt.subplots(figsize=(COL1, 2.5))
for c, s in enumerate(["A: global best", "B: per-SKU validation selection", "C: per-category selection",
                       "D1: meta-classifier", "D2: meta-weighted combination (proposed)", "Simple average of pool"]):
    reg = np.sort((EVS[s][1] - orc).values)
    ax.plot(reg, np.linspace(0, 1, len(reg)), color=PALETTE[c], label=s.split(":")[0] if ":" in s else s)
ax.set_xscale("symlog", linthresh=0.1); ax.set_xlabel("Per-SKU regret vs oracle (sMAPE pp)"); ax.set_ylabel("CDF")
ax.legend(fontsize=6); ax.set_title("Regret distribution"); fig.tight_layout(); savefig(fig, "fig_regret_cdf")

from sklearn.decomposition import PCA
Z = PHI[f][FEATS].to_numpy(float); Z = (Z - Z.mean(0)) / (Z.std(0) + 1e-9)
pc = PCA(2, random_state=0).fit(Z); XY = pc.transform(Z)
lab = LV[f].idxmin(1).to_numpy(); top = pd.Series(lab).value_counts().index[:4].tolist()
fig, ax = plt.subplots(figsize=(COL1, 2.8))
for c, m in enumerate(top):
    s = lab == m; ax.scatter(XY[s, 0], XY[s, 1], s=9, color=PALETTE[c], label=m, edgecolor="white", lw=0.3)
s = ~np.isin(lab, top); ax.scatter(XY[s, 0], XY[s, 1], s=9, color=GREY, label="other", edgecolor="white", lw=0.3)
ax.set_xlabel(f"PC1 ({pc.explained_variance_ratio_[0]:.0%})"); ax.set_ylabel(f"PC2 ({pc.explained_variance_ratio_[1]:.0%})")
ax.legend(fontsize=6); ax.set_title("Meta-feature space, coloured by validation-best model"); fig.tight_layout(); savefig(fig, "fig_feature_space")

In [ ]:
# ==== 7.4 FIGURE: META-FEATURE IMPORTANCE (mean |SHAP| of the D2 loss regressors) ====
S = sum(SHAP[f] for f in FOLD_IDS) / len(FOLD_IDS)
top = S.sum(0).sort_values(ascending=False).index[:15]
Sn = S[top].div(S[top].sum(1), axis=0)
fig, ax = plt.subplots(figsize=(COL2 * 0.7, 2.8))
im = ax.imshow(Sn.values, cmap="Blues", aspect="auto"); ax.grid(False)
ax.set_xticks(range(len(top)), top, rotation=50, ha="right"); ax.set_yticks(range(len(POOL)), POOL)
fig.colorbar(im, ax=ax, label="share of mean |SHAP|", shrink=0.8)
ax.set_title("Which series characteristics predict each model's error"); fig.tight_layout(); savefig(fig, "fig_shap_importance")
save_table(S.T.loc[top].reset_index().rename(columns={"index": "feature"}), "tab_shap", "Mean |SHAP| of meta-features in the loss regressors.", nd=4)

In [ ]:
# ==== 7.5 HETEROGENEITY: where does adaptive allocation help? (Syntetos-Boylan demand classes) ====
phi = PHI[FOLD_IDS[-1]]
sb = np.where(phi.ADI < 1.32, np.where(phi.CV2 < 0.49, "smooth", "erratic"), np.where(phi.CV2 < 0.49, "intermittent", "lumpy"))
sbs = pd.Series(sb, index=phi.index)
rows = []
for cls, g in sbs.groupby(sbs):
    r = dict(SB_class=cls, n=len(g))
    for s in ["A: global best", "B: per-SKU validation selection", "D2: meta-weighted combination (proposed)", "Oracle (test-set selection)"]:
        r[s.split(":")[0] if ":" in s else "Oracle"] = EVS[s][1].loc[g.index].mean()
    rows.append(r)
save_table(pd.DataFrame(rows), "tab_by_sb_class", "Test sMAPE by Syntetos-Boylan class of the price-change process.", nd=3)

In [ ]:
# ==== 8. ASSET INDEX + RESULTS SUMMARY ======================================
b = MAIN.set_index("strategy")
summary = f"""# Paper 3 - results summary ({time.strftime('%Y-%m-%d')})
* {N} active food SKUs; folds {FOLD_IDS}; pool of {len(POOL)} models.
* Best non-oracle strategy by sMAPE: {b.drop(index='Oracle (test-set selection)').sMAPE.idxmin()}.
* D2 sMAPE {b.loc['D2: meta-weighted combination (proposed)', 'sMAPE']:.3f} | A {b.loc['A: global best', 'sMAPE']:.3f} | B {b.loc['B: per-SKU validation selection', 'sMAPE']:.3f} | oracle {b.loc['Oracle (test-set selection)', 'sMAPE']:.3f}
"""
(PAPER_DIR / "RESULTS_SUMMARY.md").write_text(summary); print(summary)
print("figures:", sorted(p.name for p in FIG_DIR.glob("*.pdf"))); print("tables:", sorted(p.name for p in TAB_DIR.glob("*.tex")))